# Pydantic 인자와 반환 스키마

허용 값과 개수 범위를 실행 전에 검사합니다.

## 준비

In [1]:
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = Path("data/orders.csv")
assert data_path.exists(), "노트북은 labs/part5/ch19-model-tools에서 실행합니다."
print("Python 실습 준비 완료")

Python 실습 준비 완료


In [2]:
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")

## 스키마와 정상 실행

In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool
class OrderQuery(BaseModel):
    status: Literal["paid", "pending", "cancelled"] = Field(description="조회할 주문 상태")
    limit: int = Field(default=3, ge=1, le=10, description="반환할 최대 주문 수")
@tool(args_schema=OrderQuery)
def search_orders(status: str, limit: int = 3) -> list[dict]:
    """주문 상태로 주문을 조회하고 최대 limit개를 반환합니다."""
    return fetch_orders(data_path, status=status)[:limit]
print(search_orders.args_schema.model_json_schema())
print(search_orders.invoke({"status": "paid", "limit": 2}))

{'properties': {'status': {'description': '조회할 주문 상태', 'enum': ['paid', 'pending', 'cancelled'], 'title': 'Status', 'type': 'string'}, 'limit': {'default': 3, 'description': '반환할 최대 주문 수', 'maximum': 10, 'minimum': 1, 'title': 'Limit', 'type': 'integer'}}, 'required': ['status'], 'title': 'OrderQuery', 'type': 'object'}
[{'order_id': 'O101', 'customer_id': 'C001', 'status': 'paid', 'amount': 32000}, {'order_id': 'O103', 'customer_id': 'C001', 'status': 'paid', 'amount': 27000}]


## 반환 형식

In [4]:
value = search_orders.invoke({"status": "paid", "limit": 1})
print({"type": type(value).__name__, "fields": sorted(value[0])})

{'type': 'list', 'fields': ['amount', 'customer_id', 'order_id', 'status']}


## 실패 재현

In [5]:
from pydantic import ValidationError
for invalid in [{"status": "shipped", "limit": 1}, {"status": "paid", "limit": 0}]:
    try:
        search_orders.invoke(invalid)
    except ValidationError as error:
        item = error.errors()[0]
        print({"input": invalid, "type": item["type"], "loc": item["loc"]})

{'input': {'status': 'shipped', 'limit': 1}, 'type': 'literal_error', 'loc': ('status',)}
{'input': {'status': 'paid', 'limit': 0}, 'type': 'greater_than_equal', 'loc': ('limit',)}


## 수정

In [6]:
print(search_orders.invoke({"status": "pending", "limit": 1}))

[{'order_id': 'O102', 'customer_id': 'C002', 'status': 'pending', 'amount': 18000}]
